# Noise Sweep — CAST 뉴런의 환경 적응성 분석

PPO_GRU 정책(2.40M)에서, 환경 변동성(noise)이 커질수록:
1. **CAST 담당 top-k 뉴런이 얼마나 재구성되나** (clean 대비 overlap)
2. **CAST 뉴런을 끄면 성공률이 어떻게 변하나** (고정 뉴런 vs 각 환경 자기 뉴런)

를 본다. 모델·경로는 절대경로 고정. 셀을 위에서 아래로 실행.
**중요: 모든 평가는 stochastic** (deterministic은 0% 성공 — 약신호 탐색이 필수).

## 1. 셋업 (repo 루트로 chdir + import)

In [19]:
import os, sys, json
from pathlib import Path

# repo 루트 = src/ 가 있는 디렉토리. 로컬/Colab 공용.
if os.path.isdir('/content'):
    REPO_DIR = '/content/2d-osl'
else:
    REPO_DIR = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(REPO_DIR,'src')) and REPO_DIR != os.path.dirname(REPO_DIR):
        REPO_DIR = os.path.dirname(REPO_DIR)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
print('repo:', REPO_DIR)

import numpy as np
import matplotlib.pyplot as plt
from src.envs.osl_env import OslEnv
from Analysis.osl2d import eval_dump, phase4_ablation
from Analysis.osl2d._io import load_traces
from Analysis.osl2d.probe import probe_weights_episode
from Analysis.osl2d.segment import LABELS
from Analysis.osl2d.eval_dump import _env_config_from_ckpt, _resolve_ckpt
from Analysis.osl2d.policy_adapter import Policy2DAdapter
print('imports OK')

repo: /home/hyunseo/Personal_Research/OSL
imports OK


## 2. 설정 (여기만 바꾸면 됨)

In [20]:
# ===== 분석 대상 모델 (절대경로) =====
RUN_DIR = Path('/home/hyunseo/Personal_Research/OSL/runs/ppo_gru_nb_20260531_113633')
CKPT_LABEL = 'final'              # ckpt_final.pt

# ===== sweep 파라미터 =====
NOISE_STAGE = 2                   # 2 = dynamic bump field (시공간 변동)
NOISE_STRENGTHS = [0.0, 0.1, 0.2, 0.3]
SEEDS = (0, 1, 2, 3)
EPISODES_PER_SEED = 25            # trace dump용 (top-k 계산)
ABLATION_EPS = 8                 # ablation 성공률 추정 rollout/seed
MAX_STEPS = 1200
TOP_K = 16
DEVICE = 'cpu'

OUT = RUN_DIR / 'analysis' / 'noise_sweep'
OUT.mkdir(parents=True, exist_ok=True)
print('model:', RUN_DIR.name, '| strengths:', NOISE_STRENGTHS)

model: ppo_gru_nb_20260531_113633 | strengths: [0.0, 0.1, 0.2, 0.3]


## 3. 헬퍼: trace dump (stochastic) + top-k 뉴런 계산

각 noise 조건에서 stochastic trace를 뽑아 `{label}__stoch`에 저장(캐시),
선형 probe `|W|`로 행동별 top-k 뉴런을 구한다. ckpt는 항상 `final` 로드.

In [21]:
def strength_tag(a): return f'n{int(round(a*100)):02d}'

def dump_traces(strength, trace_label):
    ckpt = _resolve_ckpt(RUN_DIR, CKPT_LABEL)
    ad = Policy2DAdapter.from_checkpoint(ckpt, device=DEVICE)
    env_cfg = _env_config_from_ckpt(ckpt, NOISE_STAGE, strength)
    od = RUN_DIR/'analysis'/'traces'/trace_label; od.mkdir(parents=True, exist_ok=True)
    gi = {k:[int(i) for i in v] for k,v in ad.node_group_indices.items()}
    gi['state_size']=int(ad.n_nodes if ad.feature_dim>1 else ad.state_size)
    gi['feature_dim']=int(ad.feature_dim); gi['backbone']=ad.backbone_kind
    (od/'group_indices.json').write_text(json.dumps(gi,indent=2))
    for s in SEEDS:
        env=OslEnv(env_cfg)
        for e in range(EPISODES_PER_SEED):
            tr=eval_dump.rollout(ad,env,10000+int(s)*1000+e,MAX_STEPS,stochastic=True)
            np.savez_compressed(od/f'eval_seed{s}_ep{e:03d}.npz', **tr, episode=e,
                seed=int(s), episode_id=int(s)*10000+e, ckpt_label=trace_label,
                action_mode='stochastic')

def collect_topk(strength, force=False):
    label=f'{CKPT_LABEL}_{strength_tag(strength)}__stoch'
    td=RUN_DIR/'analysis'/'traces'/label
    if force or not (td.exists() and any(td.glob('eval_seed*_ep*.npz'))):
        print(f'  dumping traces @ {strength} ...'); dump_traces(strength,label)
    else:
        print(f'  using cached traces @ {strength}')
    tr=load_traces(RUN_DIR,[label])
    W,classes=probe_weights_episode(tr.h, tr.label.astype(int), tr.episode_id, seed=0)
    contrib=np.abs(W)
    if contrib.shape[0]==1 and len(classes)==2: contrib=np.vstack([contrib,contrib])
    tops={}
    for ri,cls in enumerate(classes):
        if ri>=contrib.shape[0]: break
        nm=LABELS[int(cls)] if int(cls)<len(LABELS) else str(cls)
        tops[nm]=np.argsort(-contrib[ri])[:TOP_K].astype(int).tolist()
    # 행동 성공률(stochastic trace 기준)
    succ=np.mean([int(np.load(f,allow_pickle=True)['success']) for f in td.glob('eval_seed*_ep*.npz')])
    return tops, float(succ)
print('helpers ready')

helpers ready


## 4. 각 noise에서 top-k 계산 + CAST overlap

In [22]:
FORCE_REDUMP = False   # True로 하면 캐시 무시하고 다시 dump

topk_by_noise, raw_succ = {}, {}
for a in NOISE_STRENGTHS:
    print(f'[noise {a}]')
    topk_by_noise[a], raw_succ[a] = collect_topk(a, force=FORCE_REDUMP)

clean_cast = topk_by_noise[NOISE_STRENGTHS[0]]['CAST']
def jaccard(x,y): x,y=set(x),set(y); return len(x&y)/max(1,len(x|y))
overlap = {a: jaccard(clean_cast, topk_by_noise[a].get('CAST',[])) for a in NOISE_STRENGTHS}

print('\nCAST top-k overlap vs clean:')
for a in NOISE_STRENGTHS:
    print(f'  noise {a}: overlap {overlap[a]:.2f}  (raw success {raw_succ[a]:.2f})')

[noise 0.0]
  using cached traces @ 0.0
[noise 0.1]
  using cached traces @ 0.1
[noise 0.2]
  using cached traces @ 0.2
[noise 0.3]
  using cached traces @ 0.3

CAST top-k overlap vs clean:
  noise 0.0: overlap 1.00  (raw success 0.84)
  noise 0.1: overlap 0.68  (raw success 0.84)
  noise 0.2: overlap 0.28  (raw success 0.81)
  noise 0.3: overlap 0.10  (raw success 0.54)


## 5. CAST ablation: 고정(clean 뉴런) vs 적응(각 환경 자기 뉴런)

- **FIXED**: clean에서 찾은 cast 뉴런을 모든 noise에서 끈다 → '같은 뉴런'의 인과 효과 변화
- **ADAPTIVE**: 각 noise의 자기 cast 뉴런을 끈다 → '그 환경의 cast 모듈'의 인과 효과

## 6. 그래프 ① — overlap (cast 모듈이 환경따라 흩어지나)

**메인 질문**: clean에서 찾은 각 행동의 top-k 뉴런이, noise가 커지면 얼마나 유지되나?
overlap이 떨어지면 = *같은 행동을 다른 뉴런이 담당* = 모듈 재구성.
CAST뿐 아니라 모든 행동을 같이 봐서 'cast만 유독 흩어지나, 다 흩어지나'를 구분한다.

In [23]:
# Per-behavior overlap of top-k neurons vs the clean condition.
BEHAVIORS = ['RUN','TURN','CAST','SPIN','STOP']
clean_top = topk_by_noise[NOISE_STRENGTHS[0]]
ov_all = {b: [jaccard(clean_top.get(b,[]), topk_by_noise[a].get(b,[])) for a in NOISE_STRENGTHS]
          for b in BEHAVIORS}

fig,ax=plt.subplots(figsize=(7,4.6))
for b in BEHAVIORS:
    style = '-o' if b=='CAST' else '--.'
    lw = 2.6 if b=='CAST' else 1.2
    ax.plot(NOISE_STRENGTHS, ov_all[b], style, lw=lw, label=b, alpha=(1.0 if b=='CAST' else 0.55))
ax.set_xlabel('Plume noise strength (stage 2, dynamic)')
ax.set_ylabel('Top-k neuron overlap vs clean (Jaccard)')
ax.set_ylim(0,1.05)
ax.set_title('Top-k neuron overlap vs clean, per behavior, across noise levels')
ax.legend(title='Behavior'); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(OUT/'overlap_all_behaviors.png',dpi=150,bbox_inches='tight'); plt.show()
print('CAST overlap:', [round(x,2) for x in ov_all['CAST']])

CAST overlap: [1.0, 0.68, 0.28, 0.1]


/tmp/ipykernel_32632/460220940.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUT/'overlap_all_behaviors.png',dpi=150,bbox_inches='tight'); plt.show()


In [24]:
# Ablate each behavior's own top-k neurons at each noise level -> Δ success rate.
def ablate_set(strength, neurons):
    ckpt=_resolve_ckpt(RUN_DIR,CKPT_LABEL)
    ad=Policy2DAdapter.from_checkpoint(ckpt,device=DEVICE)
    env=OslEnv(_env_config_from_ckpt(ckpt,NOISE_STAGE,strength))
    def run(patch):
        r=[phase4_ablation._rollout(env,ad,10000+int(s)*1000+e,patch_indices=patch,
            max_steps=MAX_STEPS,stochastic=True) for s in SEEDS for e in range(ABLATION_EPS)]
        return float(np.mean([x['success'] for x in r]))
    base=run(None)
    return {b: run(np.asarray(topk_by_noise[strength].get(b,[]),dtype=np.int64))-base
            for b in BEHAVIORS if topk_by_noise[strength].get(b)}, base

per_beh_delta, baselines = {}, {}
for a in NOISE_STRENGTHS:
    print(f'[per-behavior ablation] noise {a} ...')
    per_beh_delta[a], baselines[a] = ablate_set(a, None)

fig,ax=plt.subplots(figsize=(7.6,4.6))
for b in BEHAVIORS:
    ys=[per_beh_delta[a].get(b, np.nan) for a in NOISE_STRENGTHS]
    style='-o' if b in ('CAST','RUN','STOP') else '--.'
    ax.plot(NOISE_STRENGTHS, ys, style, lw=2.2 if b=='CAST' else 1.3, label=b,
            alpha=(1.0 if b in ('CAST','RUN','STOP') else 0.55))
ax.axhline(0,color='k',lw=0.6)
ax.set_xlabel('Plume noise strength (stage 2, dynamic)')
ax.set_ylabel('Δ success rate when behavior neurons ablated')
ax.set_title("Success drop from ablating each behavior's top-k neurons vs noise")
ax.legend(title='Ablated module'); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(OUT/'per_behavior_ablation.png',dpi=150,bbox_inches='tight'); plt.show()
print('baselines:', {a:round(baselines[a],2) for a in NOISE_STRENGTHS})

[per-behavior ablation] noise 0.0 ...


[per-behavior ablation] noise 0.1 ...
[per-behavior ablation] noise 0.2 ...
[per-behavior ablation] noise 0.3 ...
baselines: {0.0: 0.88, 0.1: 0.88, 0.2: 0.78, 0.3: 0.5}


/tmp/ipykernel_32632/108904908.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUT/'per_behavior_ablation.png',dpi=150,bbox_inches='tight'); plt.show()


## 7. 결과 저장 (json)

In [28]:
results={'run_dir':str(RUN_DIR),'noise_stage':NOISE_STAGE,'strengths':NOISE_STRENGTHS,
    'top_k':TOP_K,'clean_cast_neurons':clean_cast,
    'cast_topk_overlap_vs_clean':overlap,
    'cast_topk_per_noise':{a:topk_by_noise[a].get('CAST',[]) for a in NOISE_STRENGTHS},
    'raw_success':raw_succ,
    'per_behavior_ablation':per_beh_delta,'ablation_baselines':baselines,
    'per_behavior_overlap':ov_all}
(OUT/'noise_sweep_cast.json').write_text(json.dumps(results,indent=2,default=float))
print('saved', OUT/'noise_sweep_cast.json')

saved /home/hyunseo/Personal_Research/OSL/runs/ppo_gru_nb_20260531_113633/analysis/noise_sweep/noise_sweep_cast.json


## 8. Shared-axis PCA — do behavior clusters move across noise levels?

UMAP is nonlinear and non-deterministic: its layout changes wholesale between
conditions, so "different shape" doesn't mean "different structure" — you can't
compare panels. Instead we fit **one PCA on the pooled top-k activations across all
noise levels**, then project each noise level onto those **fixed axes**. Same
coordinate frame → we can read off whether the behavior clusters keep their shape
and only shift position (structure preserved) or actually collapse.

In [27]:
from sklearn.decomposition import PCA

LABEL_COLORS={0:'tab:blue',1:'tab:orange',2:'tab:green',3:'tab:red',4:'tab:purple'}  # RUN,TURN,CAST,SPIN,STOP
SUBSPACE_FROM_CLEAN = True   # neuron subspace fixed to clean top-k (recommended for comparison)
MAX_PTS_PER_NOISE = 3000

# 1) neuron subspace = union of clean top-k across behaviors (fixed across noise)
src_top = clean_top if SUBSPACE_FROM_CLEAN else None
def subspace_for(a):
    t = (clean_top if SUBSPACE_FROM_CLEAN else topk_by_noise[a])
    return sorted(set(i for b in BEHAVIORS for i in t.get(b,[])))

# 2) collect activations per noise on that subspace
acts, labs = {}, {}
for a in NOISE_STRENGTHS:
    tr = load_traces(RUN_DIR, [f'{CKPT_LABEL}_{strength_tag(a)}__stoch'])
    neurons = subspace_for(a)
    X = tr.h[:, neurons]; L = tr.label
    if len(X) > MAX_PTS_PER_NOISE:
        idx = np.random.RandomState(0).choice(len(X), MAX_PTS_PER_NOISE, replace=False)
        X, L = X[idx], L[idx]
    acts[a], labs[a] = X, L

# 3) ONE PCA fit on the pooled activations -> shared axes
pooled = np.vstack([acts[a] for a in NOISE_STRENGTHS])
pca = PCA(n_components=2, random_state=0).fit(pooled)
evr = pca.explained_variance_ratio_

# 4) project each noise onto the SAME axes; shared limits for fair comparison
emb = {a: pca.transform(acts[a]) for a in NOISE_STRENGTHS}
allxy = np.vstack(list(emb.values()))
xlim = (allxy[:,0].min(), allxy[:,0].max()); ylim = (allxy[:,1].min(), allxy[:,1].max())

fig, axes = plt.subplots(1, len(NOISE_STRENGTHS), figsize=(4.2*len(NOISE_STRENGTHS), 4.4), squeeze=False)
for j,a in enumerate(NOISE_STRENGTHS):
    ax=axes[0,j]; e=emb[a]; L=labs[a]
    for cls in np.unique(L):
        m=L==cls
        ax.scatter(e[m,0], e[m,1], s=4, c=LABEL_COLORS.get(int(cls),'gray'),
                   label=LABELS[int(cls)] if int(cls)<len(LABELS) else str(cls), alpha=.5)
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'noise {a}   (overlap {overlap[a]:.2f})')
axes[0,0].legend(markerscale=3, fontsize=8, loc='best')
fig.suptitle(f'Shared-axis PCA of top-k neuron activations, colored by behavior, per noise level '
             f'(PC1+PC2 = {evr.sum()*100:.0f}% var)', y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(OUT/'topk_pca_shared_by_noise.png', dpi=150, bbox_inches='tight'); plt.show()
print('saved', OUT/'topk_pca_shared_by_noise.png')

saved /home/hyunseo/Personal_Research/OSL/runs/ppo_gru_nb_20260531_113633/analysis/noise_sweep/topk_pca_shared_by_noise.png


/tmp/ipykernel_32632/2915374095.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUT/'topk_pca_shared_by_noise.png', dpi=150, bbox_inches='tight'); plt.show()
